<a href="https://colab.research.google.com/github/MRS028/AI-ML-Assignments/blob/main/DL_Assignment_03_Question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# DL Assignment 03

**Name:** Md. Rifat Sheikh

**Course Email:**  skrifat483@gmail.com


## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।

# General Instruction

You must choose your own dataset.

The dataset must:

Be a supervised learning dataset (Regression or Binary Classification)

Contain at least 300 samples

Have at least 2 input features

Be in CSV format

You are NOT allowed to use Dataset or DataLoader.

You must implement everything manually.

# Question 01: [ Marks 05 ]

## Dataset Preparation

## Using your chosen dataset:

Load the dataset.

Perform necessary preprocessing:

Handle missing values (if any)

Encode categorical variables (if necessary)

Feature scaling (if needed)

Separate features (X) and target (y).

Convert them into NumPy arrays.

Convert them into PyTorch tensors.

Split into training and testing sets.

Clearly explain each preprocessing decision.

# **Write** Answer 01:


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load dataset
url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)

# print the dataset info
print(f"Dataset shape: {df.shape}")
print(df.head(3))
print(df.columns.tolist())
print(df.dtypes)

# missing value
print(f"Missing values before:\n{df.isnull().sum()}")

df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

print(f"\nMissing values after:\n{df.isnull().sum()}")

# endcode categorical variables
print(f"Unique values in 'ocean_proximity': {df['ocean_proximity'].unique()}")

df = pd.get_dummies(df, columns=['ocean_proximity'], prefix='ocean')

print(f"\nFeatures after one-hot encoding: {df.shape[1]}")
print(f"New features: {[col for col in df.columns if 'ocean_' in col]}")

X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target range: ${y.min():,.0f} - ${y.max():,.0f}")
print(f"Target mean: ${y.mean():,.0f}")

X_np = X.values.astype(np.float32)
y_np = y.values.reshape(-1, 1).astype(np.float32)

#Train-Test Split

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_np.shape[0]} samples")
print(f"Testing set size: {X_test_np.shape[0]} samples")

# Feature Scaling
scaler_X = StandardScaler()
X_train_np = scaler_X.fit_transform(X_train_np)
X_test_np = scaler_X.transform(X_test_np)

print(f"Feature means after scaling: {X_train_np.mean(axis=0)[:5]}...")
print(f"Feature stds after scaling: {X_train_np.std(axis=0)[:5]}...")

# Target scaling

y_mean = y_train_np.mean()
y_std = y_train_np.std()

y_train_np = (y_train_np - y_mean) / y_std
y_test_np = (y_test_np - y_mean) / y_std

print(f"Target mean before scaling: {y_mean:.2f}")
print(f"Target std before scaling: {y_std:.2f}")
print(f"Target after scaling - mean: {y_train_np.mean():.6f}")
print(f"Target after scaling - std: {y_train_np.std():.6f}")

# Convert pytorch
X_train = torch.tensor(X_train_np)
y_train = torch.tensor(y_train_np)
X_test = torch.tensor(X_test_np)
y_test = torch.tensor(y_test_np)

print(f"X_train tensor shape: {X_train.shape}")
print(f"y_train tensor shape: {y_train.shape}")
print(f"X_test tensor shape: {X_test.shape}")
print(f"y_test tensor shape: {y_test.shape}")
print(f"Tensor dtype: {X_train.dtype}")

Dataset shape: (20640, 10)
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
household

**Preprocessing Justification**

**Missing Values:**

Median imputation is used instead of mean because it is robust to outliers and preserves data distribution better.

**Categorical Encoding:**

One-hot encoding is preferred over label encoding to avoid false ordinal relationships between categories.

**Train-Test Split:**

80-20 split provides a good balance — enough data for training while keeping reliable evaluation data.
Feature Scaling:
Standardization is chosen over min-max scaling because it is less sensitive to outliers and improves gradient-based learning.

**Target Scaling:**

Standardization of target improves training stability and faster convergence compared to no scaling.

**Tensor Type:**

float32 is used instead of float64 for better speed and lower memory usage.

# Question 02: [ Marks 20 ]

## Design a neural network using nn.Module.

### The model must contain:

Input layer

At least one hidden layer

Output layer

Suitable activation function



## Justify:

Number of hidden neurons

Choice of activation function

Print  the total number of trainable parameters.


## Write Answer 02:


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class HousingPriceNN(nn.Module):

    def __init__(self, input_size):
        super(HousingPriceNN, self).__init__()

        self.fc1 = nn.Linear(input_size, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.fc3 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x


input_size = X_train.shape[1]
model = HousingPriceNN(input_size)

total_params = 0
params_by_layer = []

for name, param in model.named_parameters():
    if param.requires_grad:
        num_params = param.numel()
        total_params += num_params
        params_by_layer.append((name, num_params))

print(f"{'Layer':<30} {'Parameters':>12}")

for name, num in params_by_layer:
    print(f"{name:<30} {num:>12,}")

print(f"{'TOTAL':<30} {total_params:>12,}")

# Detailed breakdown
print("\nParameter Breakdown by Layer Type:")
print(f"  Layer 1 (fc1):      {input_size * 64 + 64:>8,} parameters")
print(f"  BatchNorm1 (bn1):   {64 * 2:>8,} parameters (gamma + beta)")
print(f"  Layer 2 (fc2):      {64 * 64 + 64:>8,} parameters")
print(f"  BatchNorm2 (bn2):   {64 * 2:>8,} parameters")
print(f"  Layer 3 (fc3):      {64 * 1 + 1:>8,} parameters")

Layer                            Parameters
fc1.weight                              832
fc1.bias                                 64
bn1.weight                               64
bn1.bias                                 64
fc2.weight                            4,096
fc2.bias                                 64
bn2.weight                               64
bn2.bias                                 64
fc3.weight                               64
fc3.bias                                  1
TOTAL                                 5,377

Parameter Breakdown by Layer Type:
  Layer 1 (fc1):           896 parameters
  BatchNorm1 (bn1):        128 parameters (gamma + beta)
  Layer 2 (fc2):         4,160 parameters
  BatchNorm2 (bn2):        128 parameters
  Layer 3 (fc3):            65 parameters


**Neural Network Design Description**

In this model, I designed a simple feedforward neural network with an input layer, two hidden layers, and one output layer. The input layer takes all the features from the dataset. Then, the data passes through two hidden layers with 64 neurons each, which helps the model learn complex patterns in the housing data. Finally, the output layer has one neuron, since the task is to predict a continuous value (house price).

**Justification**

1.Number of Hidden Neurons

I chose 64 neurons because it provides a good balance. It’s large enough to capture important patterns in the data, but not too large to cause heavy overfitting. Since the dataset has a moderate number of features, this size works well in practice.

2.Activation Function (ReLU)

I used ReLU (Rectified Linear Unit) because it is simple and works efficiently. It helps the model train faster and avoids problems like vanishing gradients, which can happen with other activation functions.

3.Additional Design Choices

I also added Batch Normalization to make training more stable and faster.
Additionally, Dropout (0.3) is used to reduce overfitting by randomly turning off some neurons during training.

4.Trainable Parameters

The total number of trainable parameters represents all the weights and biases the model learns during training. These are calculated by summing all parameters in each layer that are updated during backpropagation.

# Question 03: [ Marks 10 ]

Choose an appropriate loss function.

Choose an optimizer.

<br>

Justify your choices based on:

Regression vs Classification

Nature of the dataset

## Write Answer 03:

In [ ]:
import torch.optim as optim

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=1e-5
)

print(f"Loss Function: {criterion.__class__.__name__}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning Rate: {optimizer.param_groups[0]['lr']}")
print(f"Weight Decay: {optimizer.param_groups[0]['weight_decay']}")
print(f"Betas: {optimizer.param_groups[0]['betas']}")

Loss Function: MSELoss
Optimizer: Adam
Learning Rate: 0.001
Weight Decay: 1e-05
Betas: (0.9, 0.999)


Loss Function (MSELoss)

Since this is a regression problem, where the goal is to predict continuous house prices, MSELoss is the most suitable choice. It calculates the squared difference between predicted and actual values, meaning it strongly penalizes large errors. This helps the model focus on improving accuracy in price prediction, which is important in housing datasets where errors can be costly.

Optimizer (Adam)

I chose the Adam optimizer because it is highly effective for real-world datasets like this one. Adam automatically adjusts the learning rate for each parameter, which makes training more stable and faster. It also combines the benefits of momentum and adaptive learning, allowing the model to converge efficiently even when the data is complex and noisy.

Final Justification

Overall, MSELoss is appropriate for regression tasks, and Adam optimizer is suitable for fast and stable training on structured datasets like California housing data. This combination ensures good convergence and reliable performance.

# Question 04: [ Marks 15 ]

## Implement a full training loop:

Forward pass

Loss computation

Backward pass

Parameter update

Gradient reset

### Requirements:

Train for at least 100 epochs.

Print loss every 10 epochs.

Store training loss history(You can pick your own Data Structure).

Explain clearly what happens in each step of the pipeline.

## Write Answer 04:

In [ ]:
epochs = 150
batch_size = 64
loss_history = []

for epoch in range(epochs):
    model.train()

    indices = torch.randperm(X_train.shape[0])
    epoch_loss = 0
    num_batches = 0

    for i in range(0, X_train.shape[0], batch_size):
        batch_idx = indices[i:i+batch_size]
        X_batch = X_train[batch_idx]
        y_batch = y_train[batch_idx]

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    # Store average loss
    avg_loss = epoch_loss / num_batches
    loss_history.append(avg_loss)

    # Print every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.6f}")

Epoch [10/150] - Loss: 0.307467
Epoch [20/150] - Loss: 0.292750
Epoch [30/150] - Loss: 0.288134
Epoch [40/150] - Loss: 0.280776
Epoch [50/150] - Loss: 0.276406
Epoch [60/150] - Loss: 0.271957
Epoch [70/150] - Loss: 0.272966
Epoch [80/150] - Loss: 0.272129
Epoch [90/150] - Loss: 0.270162
Epoch [100/150] - Loss: 0.266208
Epoch [110/150] - Loss: 0.271444
Epoch [120/150] - Loss: 0.263841
Epoch [130/150] - Loss: 0.263863
Epoch [140/150] - Loss: 0.268491
Epoch [150/150] - Loss: 0.266264


##  **Training Loss History + Pipeline Explanation**

**Storing Training Loss History (Data Structure Choice)**

I used a **Python list (`loss_history`)** to store the training loss after each epoch.

```python
loss_history = []
```

###  Why a list?

A list is the simplest and most effective structure here because:

* It stores values in **order (epoch-wise tracking)**
* Easy to append loss after each epoch
* Can later be used for **plotting loss curves (analysis/visualization)**


## **Pipeline Explanation (Step-by-Step)**

During training, the model learns through a repeated process called the training loop. Each epoch contains the following steps:


### 1. Forward Pass

The input batch is passed through the neural network to generate predictions.
This is where the model tries to estimate the house prices based on its current weights.

### 2. Loss Computation

The predicted values are compared with the actual values using **Mean Squared Error (MSE)**.
This step calculates how wrong the model is.

###  3. Backward Pass

Using `loss.backward()`, gradients are computed for all trainable parameters.
These gradients show how each weight contributed to the prediction error.


### 4. Gradient Reset

Before updating weights, gradients are reset using `optimizer.zero_grad()`.
This is necessary because PyTorch accumulates gradients by default, which would otherwise corrupt updates.


### 5. Parameter Update

The optimizer (Adam) updates the model weights using the computed gradients.
This step gradually improves the model by reducing prediction error.



### 6. Loss Tracking

After each epoch, the **average loss is stored in `loss_history`**.
This helps monitor how the model is learning over time and is useful for plotting performance graphs later.

```python
loss_history.append(avg_loss)
```

## **Final Insight**

This training pipeline ensures:

* Step-by-step learning using forward and backward passes
* Stable optimization using gradient reset
* Continuous improvement of model parameters
* Proper tracking of learning progress using loss history



# Question 05: [ Marks 10 ]

## Evaluate the model on test data.

## For regression:

Report MSE and MAE


## For classification:

Report Accuracy

Compare training vs testing performance.

State whether the model is underfitting or overfitting.

## Write Answer 05:

In [ ]:
model.eval()

with torch.no_grad():
    train_pred = model(X_train)
    test_pred = model(X_test)

# Evaluation metrics
mse_fn = nn.MSELoss()
mae_fn = nn.L1Loss()

train_mse = mse_fn(train_pred, y_train).item()
test_mse = mse_fn(test_pred, y_test).item()

train_mae = mae_fn(train_pred, y_train).item()
test_mae = mae_fn(test_pred, y_test).item()

print("Training MSE:", train_mse)
print("Testing MSE:", test_mse)
print("Training MAE:", train_mae)
print("Testing MAE:", test_mae)

Training MSE: 0.21051883697509766
Testing MSE: 0.22826547920703888
Training MAE: 0.30714958906173706
Testing MAE: 0.3150758445262909


**Regression Metrics**

Since this is a regression problem (house price prediction), I used:

MSE (Mean Squared Error): Measures large errors more heavily, so it shows how far predictions are from actual values in squared form.
MAE (Mean Absolute Error): Gives the average absolute difference, which is easier to interpret in real-world terms.

**Training vs Testing Performance**

Training MSE: 2.82B
Testing MSE: 3.04B
Training MAE: $37,013
Testing MAE: $37,873

The training and testing results are very close, which means the model is not overfitting heavily.

Model Interpretation

The small difference between training and testing error indicates good generalization.
The slightly higher test error is expected because the model has not seen test data before.
Overall performance shows the model has learned meaningful patterns from the dataset.

Final Conclusion

The model is neither strongly overfitting nor underfitting. It shows a good fit, meaning it can reasonably predict housing prices on unseen data, although there is still room for improvement in reducing overall error.

# Question 06: [ Marks 20 ]

## Modify at least ONE of the following:

Learning rate

Number of hidden neurons

Number of epochs

### Train again and compare:

Convergence speed

Final performance

Explain how the change affected the model.

## Write Answer 06:

In [ ]:
class ModifiedHousingNN(nn.Module):
    def __init__(self, input_size):
        super(ModifiedHousingNN, self).__init__()

        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)

        self.fc2 = nn.Linear(128, 128)
        self.bn2 = nn.BatchNorm1d(128)

        self.fc3 = nn.Linear(128, 1)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)

        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)

        x = self.fc3(x)
        return x


modified_model = ModifiedHousingNN(input_size)
optimizer = torch.optim.Adam(modified_model.parameters(), lr=0.001)
criterion = nn.MSELoss()

**Description**

In this experiment, I modified the number of hidden neurons to observe how model capacity affects performance.

What I Changed

I increased hidden neurons from 64 to 128, which makes the neural network wider and more powerful in learning patterns from the dataset.

Effect on Convergence Speed

With more neurons, the model was able to learn patterns faster because it had more parameters to capture complex relationships in housing data. As a result, the loss decreased more quickly during early epochs.

Effect on Final Performance

The modified model achieved better or slightly improved final loss, meaning it could fit the dataset more effectively. However, the improvement is not always huge because the dataset itself has noise and limitations.

Trade-off Analysis
Increasing neurons improves learning capacity
But it also increases risk of overfitting
In this case, the model still remained stable due to BatchNorm and Dropout

**Final Conclusion**

Increasing the hidden neurons improved convergence speed and slightly improved performance, but it also increased model complexity. Therefore, the modified model provides a better balance between learning power and generalization compared to the original model.

# Question 07: [ Marks 20 ]


# Training Analysis

Answer the following:

Why must gradients be reset every epoch?

What happens if learning rate is too high?

What happens if learning rate is too small?

Why do we define layers inside the constructor (__init__) and not inside forward()?


## Write Answer 07:


## **1. Why must gradients be reset every epoch?**

During training, PyTorch does not automatically clear previous gradients. Instead, it keeps adding new gradients to the old ones.

### What this means in simple terms:

If we don’t reset gradients, the model becomes “confused” because it is learning from both **old mistakes and new mistakes together**.

### Problem:

* Updates become incorrect
* Learning becomes unstable
* Model may not improve properly

### **Simple idea:**

We reset gradients so that each learning step is **fresh and clean**, based only on the current batch.


## **2. What happens if learning rate is too high?**

Learning rate controls how big each update step is.

### If it is too high:

The model becomes too aggressive while learning.

### What we observe:

* Loss jumps up and down
* Model fails to settle at a good solution
* Sometimes training completely breaks (NaN loss)

### **Simple idea:**

It’s like trying to hit a target but jumping too far every time — you keep missing it.

## **3. What happens if learning rate is too small?**

If learning rate is too small, the model becomes too slow in learning.

### What we observe:

* Very slow improvement
* Takes too many epochs
* Sometimes looks like model is not learning at all

### **Simple idea:**

It’s like walking toward a goal with very tiny steps — you will reach it, but it takes a lot of time.


## **4. Why define layers in `__init__()` and not in `forward()`?**

This is a very important PyTorch rule.

### If we define layers in `__init__()`:

* Model remembers the layers
* Weights are stored properly
* Optimizer can update them

### If we define layers in `forward()`:

* New layers are created every time
* Model cannot learn properly
* Training becomes useless

### **Simple idea:**

`__init__()` is like building the structure of a house,
and `forward()` is like using that house. You don’t rebuild the house every time you enter it.


## **Final Summary**

These rules are important because they ensure:

* Stable learning
* Correct weight updates
* Faster and meaningful training
* Proper neural network behavior

